In [197]:
import numpy as np
import pandas as pd
from scipy.stats import norm
from scipy.optimize import least_squares

In [198]:
swaption_data = pd.read_csv("../data/processed/swaption_data_clean.csv")

In [199]:
swaption_data.head()

,Expiry,Tenor,Forward,Annuity,Offset,Straddle,Strangle,RiskReversal,Payer,Receiver,Strike_Payer,Strike_Receiver,T_expiry
0,3m,1y,0.034156,0.958110,0.0025,20,10,-2,4.0,6.0,0.036656,0.031656,0.25
1,6m,1y,0.033345,0.950620,0.0050,31,13,-3,5.0,8.0,0.038345,0.028345,0.50
2,1y,1y,0.033189,0.934966,0.0100,50,18,-3,7.5,10.5,0.043189,0.023189,1.00
3,2y,1y,0.034962,0.904361,0.0100,78,42,1,21.5,20.5,0.044962,0.024962,2.00
4,3y,1y,0.036630,0.964664,0.0150,96,46,3,24.5,21.5,0.051630,0.021630,3.00


In [200]:
swaption_data['Straddle'] = swaption_data['Straddle'] / 10000
swaption_data['Strangle'] = swaption_data['Strangle'] / 10000
swaption_data['RiskReversal'] = swaption_data['RiskReversal'] / 10000
swaption_data['Payer'] = swaption_data['Payer'] / 10000
swaption_data['Receiver'] = swaption_data['Receiver'] / 10000

In [201]:
swaption_data.head()

,Expiry,Tenor,Forward,Annuity,Offset,Straddle,Strangle,RiskReversal,Payer,Receiver,Strike_Payer,Strike_Receiver,T_expiry
0,3m,1y,0.034156,0.958110,0.0025,0.0020,0.0010,-0.0002,0.00040,0.00060,0.036656,0.031656,0.25
1,6m,1y,0.033345,0.950620,0.0050,0.0031,0.0013,-0.0003,0.00050,0.00080,0.038345,0.028345,0.50
2,1y,1y,0.033189,0.934966,0.0100,0.0050,0.0018,-0.0003,0.00075,0.00105,0.043189,0.023189,1.00
3,2y,1y,0.034962,0.904361,0.0100,0.0078,0.0042,0.0001,0.00215,0.00205,0.044962,0.024962,2.00
4,3y,1y,0.036630,0.964664,0.0150,0.0096,0.0046,0.0003,0.00245,0.00215,0.051630,0.021630,3.00


## Calibrating one swaption at a time using SABR model

In [202]:
def calculate_sabr_normal_vol(F, K, T, alpha, rho, nu):
    if F == K:
        # ATM Case (K = F)
        sigma_n = alpha * (1 + ((2 - 3 * rho**2) / 24 * nu**2) * T)
    else:
        # OTM Case (K != F)
        zeta = (nu / alpha) * (F - K)
        
        # x_hat(zeta) calculation
        term = np.sqrt(1 - 2 * rho * zeta + zeta**2)
        x_hat_zeta = np.log((term + zeta - rho) / (1 - rho))
        
        # Hagan Normal Volatility Expansion
        sigma_n = alpha * (zeta / x_hat_zeta) * (1 + ((2 - 3 * rho**2) / 24 * nu**2) * T)
        
    return sigma_n

def bachelier_payer_price(F, K, T, sigma_n, annuity):
    d = (F - K) / (sigma_n * np.sqrt(T))
    price = annuity * ((F - K) * norm.cdf(d) + sigma_n * np.sqrt(T) * norm.pdf(d))
    return price

def bachelier_receiver_price(F, K, T, sigma_n, annuity):
    d = (F - K) / (sigma_n * np.sqrt(T))
    price = annuity * ((K - F) * norm.cdf(-d) + sigma_n * np.sqrt(T) * norm.pdf(d))
    return price

In [203]:
def calculate_model_prices(expiry, tenor, alpha, rho, nu):
    # Select row of swaption_data where expiry and tenor match
    swaption_data_row = swaption_data[(swaption_data['Expiry'] == expiry) & (swaption_data['Tenor'] == tenor)]

    fwd = swaption_data_row['Forward'].iloc[0]
    expiry_t = swaption_data_row['T_expiry'].iloc[0]
    annuity = swaption_data_row['Annuity'].iloc[0]

    strike_payer = swaption_data_row['Strike_Payer'].iloc[0]
    strike_receiver = swaption_data_row['Strike_Receiver'].iloc[0]
    
    # Calculating model vols
    atm_vol = calculate_sabr_normal_vol(fwd, fwd, expiry_t, alpha, rho, nu)
    payer_vol = calculate_sabr_normal_vol(fwd, strike_payer, expiry_t, alpha, rho, nu)
    receiver_vol = calculate_sabr_normal_vol(fwd, strike_receiver, expiry_t, alpha, rho, nu)

    # Calculating model prices
    atm_straddle_model_price = 2 * annuity * (atm_vol * np.sqrt(expiry_t)/np.sqrt(2*np.pi))
    payer_model_price = bachelier_payer_price(fwd, strike_payer, expiry_t, payer_vol, annuity)
    receiver_model_price = bachelier_receiver_price(fwd, strike_receiver, expiry_t, receiver_vol, annuity)
    
    return np.array([atm_straddle_model_price, payer_model_price, receiver_model_price])

In [204]:
def get_market_prices(expiry, tenor):
    swaption_data_row = swaption_data[(swaption_data['Expiry'] == expiry) & (swaption_data['Tenor'] == tenor)]
    return np.array([swaption_data_row['Straddle'].iloc[0], swaption_data_row['Payer'].iloc[0], swaption_data_row['Receiver'].iloc[0]])

In [205]:
def residuals(params, expiry, tenor):

    model_prices = calculate_model_prices(expiry, tenor, params[0], params[1], params[2])
    market_prices = get_market_prices(expiry, tenor)

    return model_prices - market_prices

In [206]:
# Loop through each row in the swaption_data DataFrame
for index, row in swaption_data.iterrows():
    expiry = row['Expiry']
    tenor = row['Tenor']

    # Initial Guesses
    initial_params = [0.00619, 0.01, 0.5]

    # Bounds: ([min_alpha, min_rho, min_nu], [max_alpha, max_rho, max_nu])
    lower_bounds = [0.0001, -0.999, 0.0001]
    upper_bounds = [0.5, 0.999, 2.0]

    # Execution
    result = least_squares(
        residuals, 
        initial_params, 
        bounds=(lower_bounds, upper_bounds),
        args=(expiry, tenor)
    )

    swaption_data.loc[(swaption_data['Expiry'] == expiry) & (swaption_data['Tenor'] == tenor), ['alpha', 'rho', 'nu']] = result.x

In [207]:
swaption_data

,Expiry,Tenor,Forward,Annuity,Offset,Straddle,Strangle,RiskReversal,Payer,Receiver,Strike_Payer,Strike_Receiver,T_expiry,alpha,rho,nu
0,3m,1y,0.034156,0.958110,0.0025,0.0020,0.0010,-0.0002,0.00040,0.00060,0.036656,0.031656,0.25,0.005122,-0.284552,1.999831
1,6m,1y,0.033345,0.950620,0.0050,0.0031,0.0013,-0.0003,0.00050,0.00080,0.038345,0.028345,0.50,0.005097,-0.241430,2.000000
2,1y,1y,0.033189,0.934966,0.0100,0.0050,0.0018,-0.0003,0.00075,0.00105,0.043189,0.023189,1.00,0.005648,-0.157938,1.525693
3,2y,1y,0.034962,0.904361,0.0100,0.0078,0.0042,0.0001,0.00215,0.00205,0.044962,0.024962,2.00,0.006058,0.029616,1.253743
4,3y,1y,0.036630,0.964664,0.0150,0.0096,0.0046,0.0003,0.00245,0.00215,0.051630,0.021630,3.00,0.005781,0.072302,0.995169
5,5y,1y,0.039854,0.816361,0.0200,0.0118,0.0056,0.0006,0.00310,0.00250,0.059854,0.019854,5.00,0.006584,0.120868,0.752189
6,10y,1y,0.046440,0.689481,0.0200,0.0135,0.0081,0.0007,0.00440,0.00370,0.066440,0.026440,10.00,0.005980,0.115446,0.603699
7,3m,2y,0.033930,1.885189,0.0025,0.0046,0.0026,-0.0002,0.00120,0.00140,0.036430,0.031430,0.25,0.005970,-0.123016,2.000000
8,6m,2y,0.033777,1.869398,0.0050,0.0070,0.0034,-0.0003,0.00155,0.00185,0.038777,0.028777,0.50,0.005841,-0.100897,2.000000
9,1y,2y,0.034062,1.837582,0.0100,0.0107,0.0042,-0.0004,0.00190,0.00230,0.044062,0.024062,1.00,0.006077,-0.095000,1.563174


In [208]:
# Save swaption_data to csv
swaption_data.to_csv("../data/processed/swaption_data_with_params.csv", index=False)